In [ ]:
# Descargamos el motor de búsqueda ligero del profesor para poder importarlo
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py

import requests 
import minsearch

# 1. Descargamos el archivo JSON con los datos del curso
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

# 2. Aplanamos los datos para tener una lista de diccionarios
documents = []
for course in documents_raw:
    course_name = course['course']
    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

# 3. Creamos el índice clásico usando minsearch
# text_fields = Las columnas donde buscará palabras
# keyword_fields = Las columnas que usaremos para filtrar exactamente (como el nombre del curso)
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

# 4. Entrenamos (llenamos) el índice con nuestros documentos
index.fit(documents)

# Imprimimos un documento de prueba para ver que todo cargó bien
print("¡Datos cargados e indexados con minsearch!")
print(documents[2])

--2026-05-10 04:00:09--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4404 (4.3K) [text/plain]
Saving to: ‘minsearch.py’

minsearch.py        100%[===================>]   4.30K  --.-KB/s    in 0s      

2026-05-10 04:00:09 (38.1 MB/s) - ‘minsearch.py’ saved [4404/4404]



/workspaces/llm-zoomcamp/02-vector-search/minsearch.py:10: UserWarning: Now minsearch is available at pypi. Run 'pip install minsearch' or 'uv add minsearch' to use it. Remove the downloaded file ('rm minsearch.py') and re-install it.
  warnings.warn(


¡Datos cargados e indexados con minsearch!
{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.", 'section': 'General course-related questions', 'question': 'Course - Can I still join the course after the start date?', 'course': 'data-engineering-zoomcamp'}


In [4]:
from openai import OpenAI

# Inicializamos el cliente de OpenAI
openai_client = OpenAI()

# 1. Función de Búsqueda (Retrieval)
def search(query):
    # Damos más peso a las coincidencias en la pregunta, y menos a la sección
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'}, # Filtro exacto
        boost_dict=boost,
        num_results=5
    )
    return results

# 2. Función para armar el Prompt (Augmented)
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    for doc in search_results:
        # Extraemos las partes útiles de cada documento y las apilamos
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    # Inyectamos la pregunta y el contexto en nuestra plantilla
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

# 3. Función para consultar al LLM (Generation)
def llm(prompt):
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 4. Función RAG maestra que une todo
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

# ¡Probamos el sistema completo con la misma pregunta del cuaderno!
print("Preguntando al sistema RAG...\n")
respuesta = rag('how do I run kafka?')
print(respuesta)

Preguntando al sistema RAG...

To run Kafka using Java, navigate to your project directory and execute the following command in the terminal:

```
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
``` 

If you are working with Python and need to run a Kafka producer, ensure you are in a virtual environment where the required packages are installed. Create the virtual environment and install dependencies by executing:

```
python -m venv env
source env/bin/activate
pip install -r ../requirements.txt
```

Remember to activate the virtual environment every time you need to work within it. For Windows, use `env\Scripts\activate` instead of `source env/bin/activate`.


In [5]:
!pip install qdrant-client fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 46.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [fastembed]14 [fastembed]ent]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from qdrant_client import QdrantClient, models

# 1. Nos conectamos al servidor local de Qdrant
qd_client = QdrantClient("http://localhost:6333")

# 2. Definimos las dimensiones y el modelo que usaremos para vectorizar
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"
collection_name = "zoomcamp-faq"

# 3. Borramos la colección si ya existía (para evitar errores si corres la celda dos veces)
qd_client.delete_collection(collection_name=collection_name)

# 4. Creamos la colección especificando que usaremos la Similitud del Coseno
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

# 5. Creamos un índice para los "payloads" (metadatos)
# Esto es crucial para poder filtrar por el nombre del curso de forma súper rápida
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

print(f"¡Colección '{collection_name}' creada exitosamente en Qdrant!")

2026-05-10 04:43:08.034293024 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


¡Colección 'zoomcamp-faq' creada exitosamente en Qdrant!


In [7]:
# --- PARTE A: Subir los documentos ---
points = []

# Iteramos sobre nuestros documentos
for i, doc in enumerate(documents):
    # Unimos la pregunta y la respuesta para tener un contexto rico
    text = doc['question'] + ' ' + doc['text']
    
    # Qdrant se encargará de crear el vector (embedding) aquí:
    vector = models.Document(text=text, model=model_handle)
    
    # Empaquetamos todo en un 'Point'
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc # Guardamos la info original
    )
    points.append(point)

print("Subiendo datos a Qdrant (Esto puede tomar un momento)...")
qd_client.upsert(
    collection_name=collection_name,
    points=points
)
print("¡Datos indexados!\n")

# --- PARTE B: La nueva función de Búsqueda Vectorial ---
def vector_search(question):
    print('vector_search is used')
    
    course = 'data-engineering-zoomcamp'
    
    # Le pedimos a Qdrant que busque los puntos más cercanos
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle 
        ),
        # Aplicamos el filtro estricto por curso (Búsqueda Híbrida)
        query_filter=models.Filter( 
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True # Queremos que nos devuelva la "mochila" con el texto
    )
    
    results = []
    # Extraemos solo los payloads de los resultados
    for point in query_points.points:
        results.append(point.payload)
    
    return results

print("Función de búsqueda vectorial lista.")

Subiendo datos a Qdrant (Esto puede tomar un momento)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

¡Datos indexados!

Función de búsqueda vectorial lista.


In [8]:
# 1. Actualizamos nuestra función maestra
def rag(query):
    # ¡Aquí está el cambio! Ahora usamos la búsqueda vectorial de Qdrant
    search_results = vector_search(query)
    
    # Reutilizamos las funciones que creamos en el Paso 2
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    
    return answer

# 2. La gran prueba final
print("Consultando a Qdrant y a OpenAI...\n")
respuesta_final = rag('how do I run kafka?')

print("\n--- RESPUESTA DEL LLM ---")
print(respuesta_final)

Consultando a Qdrant y a OpenAI...

vector_search is used

--- RESPUESTA DEL LLM ---
To run Kafka, you need to follow these steps:

1. **Ensure Kafka broker is running**: Check if your Kafka broker Docker container is active by running `docker ps`. If it's not running, navigate to the folder containing the `docker-compose.yaml` file and execute `docker compose up -d` to start all the instances.

2. **Run Producer/Consumer**:
   In the project directory, use the following command to run the Java producer:
   ```
   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
   ```
   Similarly, you can run the consumer using:
   ```
   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonConsumer.java
   ```

3. **Check configurations**: Make sure that in your scripts (like `JsonProducer.java`), the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` is set to the correct server URL and that the cluster key and secrets in `src/main/j